# Run RWE on real MIND data (RQ2 / RQ3)

Downloads **MIND-small** from Microsoft's official source (research use — the data is *not* redistributed), ingests it, learns ideological positions from click behaviour, and runs the baselines + RWE-D/RWE-B to print the paper's **RQ2** (accuracy + long-tail) and **RQ3** (ideological-diversity) tables.

Runtime: a few minutes on a free CPU runtime. Nothing is committed — only the printed metrics / `results.csv`.

> If the GitHub repo is **private**, edit the clone URL in the next cell to include a token: `https://<TOKEN>@github.com/greenwichg/random_walks_with_erasure.git`

In [ ]:
# 1) Get the code (branch with the MIND pipeline) and install it
!git clone --branch claude/sleepy-gates-oecof1 https://github.com/greenwichg/random_walks_with_erasure.git
%cd random_walks_with_erasure
!pip install -e . -q
print('installed')

In [ ]:
# 2) Download MIND-small (train) from the official source and unzip
#    (urllib raises a clear error if the URL is unreachable; -q on wget
#     hides failures, so we avoid it here)
import os, urllib.request
url = ('https://mind201910small.blob.core.windows.net/release/'
       'MINDsmall_train.zip')
print('downloading', url)
urllib.request.urlretrieve(url, 'MINDsmall_train.zip')
print('downloaded', os.path.getsize('MINDsmall_train.zip'), 'bytes (~52 MB expected)')
!unzip -q -o MINDsmall_train.zip -d MINDsmall_train && ls -lh MINDsmall_train
# If urlretrieve raises 403/404 (blob gated from Colab), upload the
# MINDsmall_train.zip you downloaded from the website instead:
#   from google.colab import files; files.upload()

In [ ]:
# 3) Ingest: click graph + political tagging + ideological positions
#    learned from clicks alone (no outlet labels). --sample-users caps
#    the dense ideal-point fit so it always fits in RAM.
import glob, os
hits = glob.glob('**/news.tsv', recursive=True)   # handles any nesting
assert hits, 'news.tsv not found - did the download/unzip succeed?'
MIND_DIR = os.path.dirname(hits[0])
print('using MIND_DIR =', MIND_DIR)
!python examples/ingest_mind.py --mind-dir {MIND_DIR} \
  --political-only --ideology --min-user-clicks 10 --min-item-clicks 10 \
  --sample-users 15000 --out mind.npz
# Watch the printed lean_corr: closer to 1.0 = the latent axis is left-right.

In [ ]:
# 4) Evaluate: baselines + RWE-D/RWE-B -> RQ2 & RQ3 tables + results.csv
#    (remove --no-bprmf to add the slower BPRMF baseline)
!python examples/eval_mind.py --npz mind.npz --out-csv results.csv --no-bprmf

In [ ]:
# 5) Show the full results table and download the CSV
import pandas as pd
df = pd.read_csv('results.csv', index_col=0)
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 50)
print(df.round(3).to_string())
try:
    from google.colab import files; files.download('results.csv')
except Exception:
    pass

## Bounded-bridging sweep (the extension's key experiment)

Vary RWE-B's *not too far* bound `d` and watch whether bridging stays strong (`uw_shift` high) while recommendations move back toward the centre (`uw_recs` low) instead of the opposite extreme (the `d=inf` row). This is the real-data test of the bounded-bridging idea in `rwe/opinion_dynamics.py`.

In [ ]:
# 6) RWE-B bounded-bridging sweep (reuses mind.npz; no re-ingest)
!python examples/eval_mind.py --npz mind.npz --out-csv sweep.csv \
  --sweep-max-distance 3,2,1.5,1,0.5 --no-bprmf
import pandas as pd
print(pd.read_csv('sweep.csv', index_col=0).round(3).to_string())

## Option B — text-grounded ideology axis (recommended)

The co-click `--ideology` axis above turns out **topical**, not left-right (check the headline eyeball). This section scores each article's lean from its **text** (title + abstract) with a pretrained classifier, uses that as the ideological axis, and re-runs eval + the sweep — so RQ3 is about real lean.

**Switch to a GPU runtime first**: Runtime -> Change runtime type -> GPU.

In [ ]:
# 7) Score political articles for lean from text
!pip install -q transformers
import glob, os
MIND_DIR = os.path.dirname(glob.glob('**/news.tsv', recursive=True)[0])
!python examples/classify_lean.py --mind-dir {MIND_DIR} --political-only --out lean.csv
# eyeball the 'Most LEFT/RIGHT-scored' headlines it prints -- they should look ideological now

In [ ]:
# 8) Re-position from text lean (no --ideology), then eval + sweep
import glob, os
MIND_DIR = os.path.dirname(glob.glob('**/news.tsv', recursive=True)[0])
!python examples/ingest_mind.py --mind-dir {MIND_DIR} --political-only \
  --positions-csv lean.csv --min-user-clicks 10 --min-item-clicks 10 \
  --sample-users 15000 --out mind_text.npz
!python examples/eval_mind.py --npz mind_text.npz --out-csv results_text.csv --no-bprmf
!python examples/eval_mind.py --npz mind_text.npz --out-csv sweep_text.csv \
  --sweep-max-distance 3,2,1.5,1,0.5 --no-bprmf
import pandas as pd
print('RESULTS (text-lean axis):')
print(pd.read_csv('results_text.csv', index_col=0).round(3).to_string())
print('\nSWEEP (text-lean axis):')
print(pd.read_csv('sweep_text.csv', index_col=0).round(3).to_string())

## Option C — how ideological is the axis? (validate it)

The text-lean axis is a noisy proxy. Quantify it: sample a few articles, label them yourself (without peeking at the model), and correlate. Or score with a second bias model and correlate the two — see `examples/validate_lean.py`.

In [ ]:
# 9) Make a 40-article labeling template, download it, label offline
import glob, os
MIND_DIR = os.path.dirname(glob.glob('**/news.tsv', recursive=True)[0])
!python examples/validate_lean.py --lean lean.csv --news-dir {MIND_DIR} --sample 40 --out label_template.tsv
from google.colab import files; files.download('label_template.tsv')
# fill the 'position' column (-1/0/1), re-upload, then run:
#   !python examples/validate_lean.py --lean lean.csv --against label_template.tsv